# P2 · Natural event-sampling baseline
Compare natural event sampling with the existing balanced E run. The event-teacher pipeline has already passed its two-song check. Enable Internet and GPU.

In [ ]:
from pathlib import Path
import subprocess, sys, os, json

project = Path("/kaggle/working/chord-practice-system")
if not project.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/PhatLavar/chord-practice-system.git", str(project)],
        check=True,
    )
os.chdir(project)
subprocess.run(["git", "pull", "--ff-only", "origin", "main"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "-e", "."], check=True)
print(subprocess.check_output(["git", "rev-parse", "HEAD"], text=True))
import torch, numpy

print("torch", torch.__version__, "numpy", numpy.__version__, "CUDA", torch.cuda.is_available())
device = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
# Official inputs. Downloaded archives are read without extraction.
from urllib.request import urlretrieve

raw = project / "data/raw"
raw.mkdir(parents=True, exist_ok=True)
sources = {
    "index.csv": "https://www.dropbox.com/s/o0olz0uwl9z9stb/billboard-2.0-index.csv?dl=1",
    "full.tar.xz": "https://www.dropbox.com/s/t390alzrkx0c9yt/billboard-2.0.1-lab.tar.xz?dl=1",
    "billboard-2.0-chordino.tar.xz": "https://www.dropbox.com/s/e9dm23vbawg9dsw/billboard-2.0-chordino.tar.xz?dl=1",
}
for name, url in sources.items():
    if not (raw / name).exists():
        temporary = raw / (name + ".part")
        urlretrieve(url, temporary)
        temporary.replace(raw / name)
cache = project / "data/processed/p1"
if not (cache / "metadata.json").exists():
    subprocess.run([sys.executable, "-m", "acr.data.billboard.prepare"], check=True)
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests"], check=True)


In [ ]:
assert torch.cuda.is_available(), "Enable a Kaggle GPU."
natural_run = project / "runs/p2-natural-kaggle"
subprocess.run(
    [
        sys.executable,
        "-m",
        "acr.training.event",
        "--config",
        "configs/p2/p2-natural.json",
        "--output",
        str(natural_run),
        "--device",
        "cuda",
    ],
    check=True,
)
print((natural_run / "metrics.json").read_text())


In [ ]:
balanced = json.loads((project / "reports/p2/kaggle-oracle/full/metrics.json").read_text())
natural = json.loads((natural_run / "metrics.json").read_text())
keys = ["event_accuracy", "rare_macro_event_recall", "macro_quality_event_recall", "best_epoch"]
comparison = {
    "seed": 20261119,
    "balanced": {key: balanced[key] for key in keys},
    "natural": {key: natural[key] for key in keys},
    "note": "Single-seed sampling comparison; inspect per-quality precision and F/E complementarity.",
}
(natural_run / "sampling_comparison.json").write_text(json.dumps(comparison, indent=2))
print(json.dumps(comparison, indent=2))
print((natural_run / "per_quality.csv").read_text())


In [ ]:
import shutil

archive = shutil.make_archive(
    "/kaggle/working/p2-natural-results", "zip", project, "runs/p2-natural-kaggle"
)
print(archive)
